In [3]:
#%load_ext autoreload
#%autoreload 2
    
import os
from gddmulde import prep
from glob import glob
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns

import functools
import operator

In [4]:
def read_file(fpath: str):
    with open(fpath, 'r') as f:
        return f.readlines()
def read_json_file(fpath: str):
    with open(fpath, 'r') as f:
        return json.load(f)
def get_records():
    db_files = glob('db-cs*.json')
    db_values = [ read_json_file(x) for x in sorted(db_files) ]
    return functools.reduce(operator.add, db_values)
def get_stdoutput_paths():
    return glob('db-cs-*.stdout')

In [5]:
!grep -H "Experiment iteration" db-cs-*.stdout | awk -F: '{last[$1] = $0} END {for (f in last) print last[f]}'

db-cs-38-47.stdout:Experiment iteration 43/100 - 1739843459288
db-cs-15-21.stdout:⏱️Experiment iteration 99/100 took 2297ms
db-cs-21-29.stdout:⏱️Experiment iteration 99/100 took 5149ms
db-cs-29-38.stdout:⏱️Experiment iteration 99/100 took 10834ms
db-cs-0-6.stdout:⏱️Experiment iteration 99/100 took 1353ms
db-cs-47-55.stdout:Experiment iteration 34/100 - 1739841474823
db-cs-6-15.stdout:⏱️Experiment iteration 99/100 took 1579ms


Check std outputs

In [8]:
fpaths_stdout = get_stdoutput_paths()
len(fpaths_stdout)
stdoutputs = dict(map(lambda x: (os.path.basename(x), read_file(x)), fpaths_stdout))
for k,v in stdoutputs.items():
    n,last = len(v),v[-1]
    print(f'{k} - n: {n} - last: {last}')

db-cs-6-15.stdout - n: 51508 - last: main() - ∆t: 1589735 [ms] (1589.735 [s])

db-cs-38-47.stdout - n: 42563 - last: Node.js v20.11.1

db-cs-29-38.stdout - n: 51508 - last: main() - ∆t: 9158924 [ms] (9158.924 [s])

db-cs-47-55.stdout - n: 19165 - last: Node.js v20.11.1

db-cs-0-6.stdout - n: 34342 - last: main() - ∆t: 864746 [ms] (864.746 [s])

db-cs-21-29.stdout - n: 45786 - last: main() - ∆t: 4380905 [ms] (4380.905 [s])

db-cs-15-21.stdout - n: 34342 - last: main() - ∆t: 1510189 [ms] (1510.189 [s])



In [9]:
df = pd.Series(get_records()).apply(pd.Series)
print('df.shape: ', df.shape)

df.shape:  (19508, 10)


---

Some agg. stats to check whether the same nr of samples have been registered for each experiment.

In [10]:
gb_keys = ['credentialSetupKey', 'cryptosuite', ]
df.groupby(gb_keys).size()[df.groupby(gb_keys).size() != 100].unstack('cryptosuite')

cryptosuite,bbs-bls-signature-2020,bbs-termwise-signature-2023,ecdsa-sd-2023-cryptosuite,ed25519-signature-2020
credentialSetupKey,,,,
mock-vc_1024-sd-256-att,43,43,43,43
mock-vc_2048-sd-016-att,34,34,34,34


Overview of the credentialSetupKey/cryptosuite configurations with less samples that the max nr. of recorded samples (assuming that the max nr. of recorded samples represents the nr of experiments)

In [11]:
df.groupby(gb_keys)['iteration'].max()[df.groupby(gb_keys)['iteration'].max()!=df.groupby(gb_keys)['iteration'].max().max()]

credentialSetupKey       cryptosuite                
mock-vc_1024-sd-256-att  bbs-bls-signature-2020         42
                         bbs-termwise-signature-2023    42
                         ecdsa-sd-2023-cryptosuite      42
                         ed25519-signature-2020         42
mock-vc_2048-sd-016-att  bbs-bls-signature-2020         33
                         bbs-termwise-signature-2023    33
                         ecdsa-sd-2023-cryptosuite      33
                         ed25519-signature-2020         33
Name: iteration, dtype: int64

In [12]:
experiment_counts = df.groupby(gb_keys).size().rename('experimentCounts').value_counts()
experiment_counts

experimentCounts
100    192
43       4
34       4
Name: count, dtype: int64

In [13]:
if experiment_counts.shape[0] != 1:
    raise Warning(f'''
Not all experiments contain the same number of samples.
Experiment counts:
{experiment_counts}

This could indicate that some experiments are still running.''')

Warning: 
Not all experiments contain the same number of samples.
Experiment counts:
experimentCounts
100    192
43       4
34       4
Name: count, dtype: int64

This could indicate that some experiments are still running.

---

In [ ]:
json.dump(records, open('data/db-cs-concat-x.json', 'w'))

----